# Celery: a map-reduce spread over several workers

This demo runs the slides' map-reduce for real on Celery. The application sends tasks to a broker; workers, separate processes, take them from there, run them and store their results in a backend, where the application reads them. The demo then compares the duration of the same batch of tasks with one worker, then with four workers behind the same broker: that's Celery's load balancing. The next cell installs Celery.

In [1]:
%pip install -q celery sqlalchemy

Note: you may need to restart the kernel to use updated packages.


## The tasks module

In production, the broker and the backend are services: Redis or RabbitMQ, for instance `broker="redis://localhost:6379/0"`. So that the demo has no service to start, locally as in Colab, it stores the messages and the results in two SQLite databases, plain files: `sqla+sqlite://` is kombu's SQLAlchemy transport (Celery's messaging library), `db+sqlite://` Celery's SQLAlchemy backend. It's slow, since an idle worker only reads the database again once a second, and it's only good for a demonstration: only the processes of the machine that sees these files can be its workers.

The `square` task waits one second (`time.sleep`) before returning its result, like a task waiting for an answer from the network; `total` will do the reduce of the map-reduce. With `worker_prefetch_multiplier = 1`, a worker reserves only one task in advance, instead of four per process by default, which another, free worker could no longer take. With `result_extended = True`, the backend also keeps, with each result, the name of the worker that computed it.

In [2]:
%%writefile tasks.py
import time

from celery import Celery

app = Celery(
  "tasks",
  broker="sqla+sqlite:///broker.sqlite",  # in production: "redis://localhost:6379/0"
  backend="db+sqlite:///results.sqlite",  # in production: "redis://localhost:6379/1"
)
app.conf.worker_prefetch_multiplier = 1  # one task reserved in advance per process
app.conf.result_extended = True  # worker name kept with each result


@app.task
def square(x: int) -> int:
  time.sleep(1)  # one second of waiting: a network request, a slow service…
  return x * x


@app.task
def total(numbers: list[int]) -> int:
  return sum(numbers)

Writing tasks.py


## Starting and stopping workers

A worker is usually started in a terminal, with `celery -A tasks worker`. Here, each worker is a subprocess of the notebook, started with the same interpreter (`sys.executable`), and its log goes to a file, `worker-1.log` for the first one. The `workers` context manager first empties the queue (`app.control.purge`, for the tasks of an interrupted run), starts `n` workers of one process each (`--concurrency=1`), waits for each to write `ready.` in its log, then stops them on leaving the `with` block, even if the block's code raises an exception.

In [3]:
import collections
import contextlib
import subprocess
import sys
import time
from pathlib import Path

from celery import chord, group

from tasks import app, square, total


@contextlib.contextmanager
def workers(n: int):
  app.control.purge()
  processes = []
  logs = [Path(f"worker-{k}.log") for k in range(1, n + 1)]
  try:
    for k, log in enumerate(logs, start=1):
      with log.open("w") as out:
        processes.append(
          subprocess.Popen(
            [sys.executable, "-m", "celery", "-A", "tasks", "worker"]
            + ["-n", f"worker{k}@demo", "--concurrency=1", "--loglevel=INFO"],
            stdout=out,
            stderr=subprocess.STDOUT,
          )
        )
    start = time.perf_counter()
    while not all("ready." in log.read_text() for log in logs):
      if time.perf_counter() - start > 60:
        raise TimeoutError(f"{n} workers not ready after 60 s, see worker-*.log")
      time.sleep(0.2)
    yield
  finally:
    for p in processes:
      p.terminate()  # warm shutdown: the worker first finishes its current task
    for p in processes:
      p.wait(timeout=30)

## A single worker

`delay` sends a task to the broker and returns an `AsyncResult` at once, whose `get` waits for the result. Then comes a `group` of 8 tasks: with a single worker of a single process, they go one after the other, in 8 seconds at least.

In [4]:
with workers(1):
  result = square.delay(4)
  print(result.get(timeout=10))

  start = time.perf_counter()
  squares = group(square.s(i) for i in range(8))()
  print(squares.get(timeout=30))
  print(f"1 worker: {time.perf_counter() - start:.1f} s")

16
[0, 1, 4, 9, 16, 25, 36, 49]
1 worker: 9.1 s


## Four workers behind the same broker

The same 8 tasks, this time with four workers reading the same broker: each takes the next task as soon as it's free, without the application knowing how many workers are running. To scale up, you therefore start more workers, on the same machine or on other machines that see the broker, without changing anything in the application. The `worker` attribute of each result (thanks to `result_extended`) shows the distribution.

In [5]:
with workers(4):
  start = time.perf_counter()
  squares = group(square.s(i) for i in range(8))()
  print(squares.get(timeout=30))
  print(f"4 workers: {time.perf_counter() - start:.1f} s")

  distribution = collections.Counter(result.worker for result in squares.results)
  for name, nb_tasks in sorted(distribution.items()):
    print(name, nb_tasks)

[0, 1, 4, 9, 16, 25, 36, 49]
4 workers: 3.3 s
worker1@demo 2
worker2@demo 2
worker3@demo 2
worker4@demo 2


## The map-reduce: `chord`

The slides' `chord`: the 8 squares of the header (the map) run in parallel on the four workers, then the `total` callback (the reduce) receives the list of their results, 0 + 1 + 4 + … + 49 = 140. With this backend, a Celery task (`chord_unlock`) checks every second whether the whole header has finished, hence about one second more than with the `group`; Redis notifies the callback as soon as the last result is in.

In [6]:
with workers(4):
  start = time.perf_counter()
  result = chord(square.s(i) for i in range(8))(total.s())
  print(result.get(timeout=30))
  print(f"chord on 4 workers: {time.perf_counter() - start:.1f} s")

140
chord on 4 workers: 4.1 s


## In production

The same code runs with a Redis or RabbitMQ broker: only the URLs of `Celery(...)` change. On one machine, a single worker of several processes replaces the four workers of this demo (`celery -A tasks worker --concurrency=4`), or adjusts their number to the length of the queue (`--autoscale=8,2`: between 2 and 8 processes). On several machines, each one starts its workers against the same broker. With Redis or RabbitMQ, you also monitor the workers: `celery -A tasks inspect active` lists the running tasks of each, and Flower (`pip install flower`, then `celery -A tasks flower`) shows them in a web dashboard. This demo's SQLite broker doesn't pass on these monitoring commands, hence reading the logs to know when the workers are ready.